In [ ]:
# PDF and PNG

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# ---------- Config ----------
CSV_PATH = "avg_metrics_per_model_per_organ.csv"  
OUTPUT_DIR = Path("reg_figures")
OUTPUT_DIR.mkdir(exist_ok=True)


# Mapping from CSV model names -> display names
display_names = {
    "HistGen": "HistGen",
    "UNI1": "UNI",
    "Conch": "Conch",
    "UNI2_4": "UNI2",
    "TITAN": "TITAN",
}


# Consistent model colors, keyed by CSV names
model_colors = {
    "HistGen": "#7f8c8d",
    "UNI1": "#3498db",
    "Conch": "#16b895",
    "UNI2_4": "#8e44ad",
    "TITAN": "#e74c3c",
}


# Fixed organ order
organ_order = [
    "Overall",
    "Breast",
    "Urinary bladder",
    "Uterine cervix",
    "Colon",
    "Lung",
    "Prostate",
    "Stomach",
    "Rectum",
]


# Desired model order, using CSV names
model_order = ["HistGen", "UNI1", "Conch", "UNI2_4", "TITAN"]


# ---------- Load data ----------
df = pd.read_csv(CSV_PATH)


# Ensure all needed metric columns exist
required_metrics = {
    "REG_AverageRanking": "REGScore",
    "METEOR": "METEOR",
    "ROUGE_L": "ROUGE-L",
    "BLEU_4": "BLEU-4",
}
for col in required_metrics.keys():
    if col not in df.columns:
        raise ValueError(f"{col} column not found in CSV")


# ---------- Helper for one metric ----------
def plot_metric_bars(metric_col, metric_label, filename_stem, dpi=300):
    """Create organ-level grouped bar chart for one metric.
    Saves both a PDF (vector, for the paper) and a PNG (raster, for README/GitHub).
    """
    pivot = df.pivot(index="Organ", columns="Model", values=metric_col)
    # keep only organs we care about, in fixed order
    orgs_in_data = pivot.index.tolist()
    ordered_orgs = [o for o in organ_order if o in orgs_in_data]
    pivot = pivot.reindex(index=ordered_orgs)
    # reorder models (columns)
    pivot = pivot.reindex(columns=model_order)

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(pivot.index))
    width = 0.8 / len(pivot.columns)

    for idx, model_key in enumerate(model_order):
        if model_key not in pivot.columns:
            continue
        color = model_colors.get(model_key, "#555555")
        ax.bar(
            x + idx * width,
            pivot[model_key].values,
            width,
            label=display_names[model_key],
            color=color,
        )

    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index, rotation=45, ha="right", fontsize=12)
    for lbl in ax.get_xticklabels():
        lbl.set_fontweight("bold")

    ax.set_ylabel(metric_label, fontsize=12, fontweight="bold")
    ax.legend(fontsize=12, prop={"weight": "bold"}, edgecolor="black")
    ax.tick_params(axis="y", labelsize=12)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")

    fig.tight_layout()

    # Save both formats
    fig.savefig(OUTPUT_DIR / f"{filename_stem}.pdf")
    fig.savefig(OUTPUT_DIR / f"{filename_stem}.png", dpi=dpi)

    plt.close(fig)


# ---------- Create all four PDF + PNG pairs ----------
plot_metric_bars("REG_AverageRanking", "REGScore", "reg_grouped_bars")
plot_metric_bars("METEOR", "METEOR", "meteor_grouped_bars")
plot_metric_bars("ROUGE_L", "ROUGE-L", "rougeL_grouped_bars")
plot_metric_bars("BLEU_4", "BLEU-4", "bleu4_grouped_bars")